# Baseline 2: Vanilla RAG Conversational QA
## GraphRAG Benchmark — Bloomberg Financial News

**Project**: GraphRAG vs Vanilla RAG vs Pure LLM  
**Role**: This notebook answers financial questions using standard vector-based RAG — chunk the articles, embed them, retrieve the most relevant chunks by query similarity, and feed those chunks as context to the LLM.

---

**What is being compared:**

| System | Context source | This notebook? |
|--------|---------------|----------------|
| GraphRAG | Subgraph retrieved from knowledge graph | No — your pipeline |
| Vanilla RAG | Top-k chunks from vector index | **This notebook** |
| Pure LLM | None — LLM memory only | Baseline 1 |

**The key research question this notebook addresses:**  
Does structuring retrieved information as a knowledge graph (GraphRAG) produce better answers than retrieving raw text chunks (Vanilla RAG)?  
Vanilla RAG retrieves semantically similar text, but it has no understanding of entity relationships across documents.  
GraphRAG retrieves a subgraph that explicitly encodes which entities are connected and how.

**prerequisite**: Run Baseline 1 first to generate `qa_eval_set.json`.  
This notebook loads that shared QA set and scores answers on the same metrics.

## 1. Environment Setup

**Additional packages vs Baseline 1:**  
- `sentence-transformers` — dense embedding model for chunk indexing  
- `faiss-cpu` — vector similarity search (CPU-based to preserve GPU VRAM for the LLM)  
- `langchain-text-splitters` — chunking with configurable overlap

In [ ]:
import subprocess, sys
packages = [
    "datasets","transformers","torch","bitsandbytes","accelerate",
    "pandas","numpy","tqdm","rouge-score",
    "sentence-transformers","faiss-cpu","langchain-text-splitters",
]
for pkg in packages:
    subprocess.check_call([sys.executable,"-m","pip","install",pkg,"-q"])
print("Setup complete.")

## 2. Imports and Configuration

**Design choices documented here for the final report:**

**Chunking — 512 chars / 64 overlap:**  
Financial sentences are information-dense. A 512-character window typically covers 2-3 complete sentences, which is enough to capture an entity-relation-entity triple.  
The 64-character overlap prevents relation spans from being cut across chunk boundaries.

**Embedding — `all-MiniLM-L6-v2`:**  
This 22M-parameter model fits on CPU and embeds 5000 articles in minutes. It is the de-facto standard for RAG benchmarks, making our results comparable to published work.

**Top-k = 5:**  
Five chunks give ~2500 characters of context — enough information to answer a factual financial question without overflowing Mistral's context window.

In [ ]:
import json, os, re, time
import numpy as np
import pandas as pd
import torch
import faiss
from tqdm import tqdm
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, pipeline
)
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from rouge_score import rouge_scorer
from collections import defaultdict

torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

CFG = {
    # Model
    "llm_name"         : "mistralai/Mistral-Nemo-Instruct-2407",
    "embed_model_name" : "sentence-transformers/all-MiniLM-L6-v2",
    "max_new_tokens"   : 400,
    "temperature"      : 0.1,
    # Data
    "dataset_name"     : "XJCEO/bloomberg_financial_news_120k",
    "n_articles"       : 5000,
    # RAG
    "chunk_size"       : 512,
    "chunk_overlap"    : 64,
    "top_k"            : 5,
    # Paths
    "qa_set_path"      : "/content/qa_eval_set.json",
    "index_path"       : "/content/rag_faiss.index",
    "output_path"      : "/content/rag_qa_results.json",
}
print("Configuration loaded.")

## 3. Load Dataset and Shared QA Evaluation Set

**Why we load the QA set from Baseline 1:**  
All three baselines must answer the exact same 25 questions so that scores are directly comparable.  
The QA set was generated in Baseline 1 from the same Bloomberg subset and saved to `/content/qa_eval_set.json`.  
If you are running this notebook in the same Colab session, the file is already at that path.  
If not, copy it from Google Drive first.

In [ ]:
# ── Load Bloomberg articles for RAG index
print("Loading Bloomberg Financial News dataset...")
dataset = load_dataset(CFG["dataset_name"], split="train")
df = dataset.to_pandas()
TEXT_COL = "Article" if "Article" in df.columns else df.select_dtypes("object").columns[0]
df = df[df[TEXT_COL].apply(lambda x: isinstance(x,str) and len(x.strip())>30)].reset_index(drop=True)
df["article_id"] = df.index
df = df.head(CFG["n_articles"]).copy()
print(f"Subset loaded: {len(df)} articles")

# ── Load shared QA set
# If running in same session as Baseline 1, file is already at /content/qa_eval_set.json
# Otherwise copy from Drive first:
#   from google.colab import drive; drive.mount('/content/drive')
#   import shutil; shutil.copy('/content/drive/MyDrive/qa_eval_set.json', '/content/')

with open(CFG["qa_set_path"]) as f:
    qa_eval_set = json.load(f)

print(f"\nQA set loaded: {len(qa_eval_set)} questions")
for qa in qa_eval_set[:5]:
    print(f"  [{qa['qa_id']:2d}] ({qa['topic']:10s}) {qa['question'][:65]}")

## 4. Article Chunking

**Why `RecursiveCharacterTextSplitter` over naive fixed-length splitting:**  
This splitter tries to break at paragraph (`\n\n`) then sentence (`.`) then word (` `) boundaries, in that priority order.  
This preserves semantic coherence within each chunk — a single chunk is unlikely to cut a company name from its associated verb.  
Naive character splitting would frequently do this, damaging retrieval quality.

Each chunk stores its `article_id` so we can trace retrieved chunks back to their source article during evaluation.

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size    = CFG["chunk_size"],
    chunk_overlap = CFG["chunk_overlap"],
    separators    = ["\n\n","\n",". "," ",""],
)

chunks   = []
chunk_id = 0

print(f"Chunking {len(df)} articles...")
for _, row in tqdm(df.iterrows(), total=len(df), desc="Chunking"):
    article_id   = int(row["article_id"])
    article_text = str(row[TEXT_COL])
    splits       = splitter.split_text(article_text)
    for split_text in splits:
        if len(split_text.strip()) < 20:
            continue
        chunks.append({
            "chunk_id"  : chunk_id,
            "article_id": article_id,
            "headline"  : str(row.get("Headline","")),
            "text"      : split_text,
        })
        chunk_id += 1

print(f"Total chunks         : {len(chunks):,}")
print(f"Avg chunks / article : {len(chunks)/len(df):.1f}")
print(f"Avg chunk length     : {np.mean([len(c['text']) for c in chunks]):.0f} chars")

## 5. Embedding and FAISS Index Construction

**Two-stage GPU strategy:**  
The embedding model runs on CPU to preserve all GPU VRAM for the LLM.  
FAISS IndexFlatIP performs exact cosine search over L2-normalised vectors — no approximation error, no ANN tuning required at this scale (< 100K chunks).

The index is saved to disk so it can be reloaded across sessions without re-embedding.

In [ ]:
print(f"Loading embedding model: {CFG['embed_model_name']}")
embed_model = SentenceTransformer(CFG["embed_model_name"], device="cpu")
EMB_DIM = embed_model.get_sentence_embedding_dimension()
print(f"Embedding dimension: {EMB_DIM}")

print(f"\nEmbedding {len(chunks):,} chunks...")
chunk_texts = [c["text"] for c in chunks]
BATCH = 256
all_embs = []

for i in tqdm(range(0, len(chunk_texts), BATCH), desc="Embedding"):
    batch = chunk_texts[i:i+BATCH]
    embs  = embed_model.encode(batch, normalize_embeddings=True, show_progress_bar=False)
    all_embs.append(embs)

embeddings = np.vstack(all_embs).astype(np.float32)
print(f"Embeddings shape: {embeddings.shape}")

index = faiss.IndexFlatIP(EMB_DIM)
index.add(embeddings)
faiss.write_index(index, CFG["index_path"])
print(f"FAISS index built  : {index.ntotal:,} vectors")
print(f"Index saved        : {CFG['index_path']}")

## 6. RAG Retrieval Function

**Retrieval query = the question itself:**  
For QA (unlike entity extraction), the question is already a precise semantic query.  
We embed the question and retrieve the top-5 most similar chunks from the full corpus.

**Why retrieve from the full 5000-article corpus, not just the source article:**  
In a real RAG deployment the system does not know which article contains the answer.  
Restricting retrieval to the correct article would be cheating.  
The retriever must find the right content on its own — this is the core challenge of RAG.

In [ ]:
def retrieve(query: str, k: int = None) -> list[dict]:
    """
    Embed query and retrieve top-k chunks from the FAISS index.
    Returns list of chunk dicts enriched with similarity score.
    """
    k = k or CFG["top_k"]
    q_emb = embed_model.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, indices = index.search(q_emb, k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < len(chunks):
            results.append({**chunks[idx], "score": float(score)})
    return results

# Sanity check
test_q  = "What did Marriott report about its profit forecast?"
test_r  = retrieve(test_q)
print(f"Test query: '{test_q}'")
print(f"Top {len(test_r)} chunks retrieved:")
for r in test_r:
    print(f"  [article {r['article_id']:4d}] score={r['score']:.3f}  {r['text'][:80]}...")

## 7. Load LLM

Same Mistral NeMo 12B with 4-bit NF4 quantization as Baseline 1.  
The model is identical — the only difference is that here we pass retrieved chunk context in the prompt.

In [ ]:
print(f"Loading {CFG['llm_name']} with 4-bit quantization...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_compute_dtype    = torch.bfloat16,
    bnb_4bit_use_double_quant = True,
)
tokenizer = AutoTokenizer.from_pretrained(CFG["llm_name"])
model = AutoModelForCausalLM.from_pretrained(
    CFG["llm_name"],
    quantization_config = bnb_config,
    device_map          = "auto",
    trust_remote_code   = True,
)
model.eval()

pipe = pipeline(
    "text-generation", model=model, tokenizer=tokenizer,
    max_new_tokens=CFG["max_new_tokens"], temperature=CFG["temperature"],
    do_sample=False,
)
vram = torch.cuda.memory_allocated()/1e9 if torch.cuda.is_available() else 0
print(f"Model loaded. VRAM: {vram:.2f} GB")

## 8. RAG Prompt Design

**How the RAG prompt differs from Pure LLM (Baseline 1):**  
The user turn includes a `[Retrieved Context]` block containing the top-5 chunks before the question.  
The LLM is explicitly instructed to ground its answer in that context.

**Why explicit grounding instruction matters:**  
Without it, the model may ignore the context and answer from memory — defeating the purpose of RAG.  
The instruction "use the provided context" is standard in production RAG systems.

**Context ordering — highest similarity first:**  
Chunks are ordered by retrieval score. The most relevant chunk appears first and is therefore closest to the question in the prompt, which tends to produce better answers for instruction-tuned models.

In [ ]:
SYSTEM_PROMPT = """You are a knowledgeable financial analyst with access to retrieved Bloomberg news articles.
Answer the question using the provided context. Be specific and cite relevant facts from the context.
If the context does not contain enough information, say so clearly and answer from general knowledge.""""

def build_rag_prompt(question: str, retrieved_chunks: list[dict]) -> str:
    """Build the RAG prompt: retrieved context + question."""
    # Build context block (highest score first — already sorted by FAISS)
    context_lines = []
    for i, chunk in enumerate(retrieved_chunks, 1):
        context_lines.append(
            f"[Source {i} | Article {chunk['article_id']} | "
            f"Similarity {chunk['score']:.3f}]\n{chunk['text']}"
        )
    context_block = "\n\n".join(context_lines)

    return f"""Retrieved context from Bloomberg Financial News:

{context_block}

Question: {question}

Answer based on the context above:"""

def ask_rag(question: str) -> tuple[str, list[dict]]:
    """Retrieve chunks and ask the LLM with that context."""
    retrieved = retrieve(question)
    prompt    = build_rag_prompt(question, retrieved)
    messages  = [
        {"role":"system","content":SYSTEM_PROMPT},
        {"role":"user","content":prompt},
    ]
    try:
        raw   = pipe(messages)[0]["generated_text"]
        reply = raw[-1].get("content","") if isinstance(raw,list) else str(raw)
        return reply.strip(), retrieved
    except Exception as e:
        return f"[ERROR: {e}]", retrieved

print("RAG QA function ready.")

## 9. RAG Inference Loop

For each of the 25 shared questions:
1. Embed the question and retrieve top-5 chunks from the FAISS index
2. Build the RAG prompt with those chunks as context
3. Run LLM inference
4. Store the answer plus metadata about what was retrieved

In [ ]:
print(f"Running Vanilla RAG inference on {len(qa_eval_set)} questions...")
print()

results = []
t_start = time.time()

for qa in tqdm(qa_eval_set, desc="RAG QA"):
    answer, retrieved = ask_rag(qa["question"])

    # Track whether the correct source article was retrieved
    source_article_id  = qa["article_id"]
    retrieved_art_ids  = [c["article_id"] for c in retrieved]
    correct_retrieved  = source_article_id in retrieved_art_ids

    results.append({
        "qa_id"              : qa["qa_id"],
        "article_id"         : qa["article_id"],
        "topic"              : qa["topic"],
        "question"           : qa["question"],
        "reference_answer"   : qa["reference_answer"],
        "predicted_answer"   : answer,
        "method"             : "vanilla_rag",
        "context_used"       : [c["text"][:100] for c in retrieved],
        "retrieved_article_ids": retrieved_art_ids,
        "top1_score"         : retrieved[0]["score"] if retrieved else 0.0,
        "correct_source_retrieved": correct_retrieved,
    })

elapsed = time.time() - t_start
print(f"\nInference done: {len(results)} answers in {elapsed:.0f}s ({elapsed/len(results):.1f}s each)")

# Retrieval hit rate
hit_rate = np.mean([r["correct_source_retrieved"] for r in results])
print(f"Source article retrieved (top-5): {hit_rate*100:.1f}% of questions")
print()

for r in results[:3]:
    print(f"Q: {r['question']}")
    print(f"A: {r['predicted_answer'][:200]}")
    print(f"Source retrieved: {r['correct_source_retrieved']} | Articles: {r['retrieved_article_ids']}")
    print()

## 10. Evaluation

**Identical metrics to Baseline 1** — ROUGE-L and LLM-as-Judge — so scores are directly comparable.

**Additional RAG-specific metric — Source Retrieval Hit Rate:**  
The percentage of questions where the correct source article appeared in the top-5 retrieved chunks.  
A high hit rate means the retriever found the right article; a low hit rate means the LLM is answering from chunks that may not contain the answer — a core weakness of Vanilla RAG that GraphRAG's structured retrieval is designed to address.

In [ ]:
rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

JUDGE_SYSTEM = """You are an objective evaluator of financial QA answers.
Score the predicted answer on three criteria (1-5 each):
- relevance: does it answer the question asked?
- factual_grounding: are the claims accurate and supported by the context provided?
- completeness: is the answer sufficiently detailed?

Output ONLY valid JSON: {"relevance": 3, "factual_grounding": 4, "completeness": 2, "reasoning": "brief explanation"}"""

def llm_judge(question:str, reference:str, predicted:str, context:list[str]) -> dict:
    context_preview = " | ".join(c[:80] for c in context[:3])
    prompt = f"""Question: {question}

Reference answer: {reference}

Retrieved context (summary): {context_preview}

Predicted answer: {predicted}

Score the predicted answer. Output ONLY the JSON:"""
    messages = [{"role":"system","content":JUDGE_SYSTEM},{"role":"user","content":prompt}]
    try:
        raw   = pipe(messages)[0]["generated_text"]
        reply = raw[-1].get("content","") if isinstance(raw,list) else str(raw)
        reply = re.sub(r"```json\s*","",reply)
        reply = re.sub(r"```\s*","",reply).strip()
        m = re.search(r"\{.*\}",reply,re.DOTALL)
        if m:
            s = json.loads(m.group())
            return {
                "relevance"        : float(s.get("relevance",0)),
                "factual_grounding": float(s.get("factual_grounding",0)),
                "completeness"     : float(s.get("completeness",0)),
                "reasoning"        : s.get("reasoning",""),
            }
    except Exception:
        pass
    return {"relevance":0,"factual_grounding":0,"completeness":0,"reasoning":"parse error"}

print(f"Evaluating {len(results)} answers (ROUGE-L + LLM judge)...")
for r in tqdm(results, desc="Evaluating"):
    rg = rouge.score(r["reference_answer"], r["predicted_answer"])
    r["rouge_l"] = round(rg["rougeL"].fmeasure, 4)
    j = llm_judge(r["question"], r["reference_answer"], r["predicted_answer"], r["context_used"])
    r["judge"]     = j
    r["judge_avg"] = round((j["relevance"]+j["factual_grounding"]+j["completeness"])/3, 4)

# ── Print results table
print()
print("=== VANILLA RAG QA EVALUATION RESULTS ===")
print(f"{'#':>3}  {'Topic':10s}  {'ROUGE-L':>8}  {'Judge Avg':>9}  {'Src Hit':>7}  Question")
print("-"*82)
for r in results:
    hit = "yes" if r["correct_source_retrieved"] else "no"
    print(f"{r['qa_id']:3d}  {r['topic']:10s}  {r['rouge_l']:8.4f}  {r['judge_avg']:9.4f}  {hit:>7}  {r['question'][:40]}")

avg_rouge = float(np.mean([r["rouge_l"]   for r in results]))
avg_judge = float(np.mean([r["judge_avg"] for r in results]))
avg_rel   = float(np.mean([r["judge"]["relevance"]          for r in results]))
avg_fact  = float(np.mean([r["judge"]["factual_grounding"]  for r in results]))
avg_comp  = float(np.mean([r["judge"]["completeness"]       for r in results]))
hit_rate  = float(np.mean([r["correct_source_retrieved"]    for r in results]))

print()
print("=== AGGREGATE METRICS ===")
print(f"  ROUGE-L (avg)             : {avg_rouge:.4f}")
print(f"  Judge score (avg)         : {avg_judge:.4f} / 5.0")
print(f"    Relevance               : {avg_rel:.4f}")
print(f"    Factual grounding       : {avg_fact:.4f}")
print(f"    Completeness            : {avg_comp:.4f}")
print(f"  Source retrieval hit rate : {hit_rate*100:.1f}%")

topic_scores = defaultdict(list)
for r in results:
    topic_scores[r["topic"]].append(r["judge_avg"])
print("\n  By topic:")
for topic, scores in sorted(topic_scores.items()):
    print(f"    {topic:12s}: {np.mean(scores):.4f}")

## 11. Save Results

In [ ]:
summary = {
    "method"               : "Vanilla RAG (Mistral NeMo 12B + MiniLM + FAISS)",
    "n_questions"          : len(results),
    "chunk_size"           : CFG["chunk_size"],
    "chunk_overlap"        : CFG["chunk_overlap"],
    "top_k"                : CFG["top_k"],
    "avg_rouge_l"          : avg_rouge,
    "avg_judge_score"      : avg_judge,
    "avg_relevance"        : avg_rel,
    "avg_factual"          : avg_fact,
    "avg_completeness"     : avg_comp,
    "source_hit_rate"      : hit_rate,
    "per_question"         : results,
}

with open(CFG["output_path"],"w") as f:
    json.dump(summary, f, indent=2)

print(f"Results saved : {CFG['output_path']}")
print(f"Size          : {os.path.getsize(CFG['output_path'])/1024:.1f} KB")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import shutil

for fpath in [CFG["output_path"], CFG["index_path"]]:
    if os.path.exists(fpath):
        dest = f"/content/drive/MyDrive/{os.path.basename(fpath)}"
        shutil.copy(fpath, dest)
        print(f"Saved to Drive: {os.path.basename(fpath)}")

## 12. Side-by-Side Preview vs Baseline 1

Before the full three-way comparison notebook, we can already preview how Vanilla RAG compares to Pure LLM by loading Baseline 1's results if available.

In [ ]:
purell_path = "/content/purell_qa_results.json"

if os.path.exists(purell_path):
    with open(purell_path) as f:
        purell = json.load(f)

    print("=" * 65)
    print("PRELIMINARY COMPARISON: Pure LLM vs Vanilla RAG")
    print("=" * 65)
    print(f"{'Metric':30s}  {'Pure LLM':>10}  {'Vanilla RAG':>12}")
    print("-" * 55)
    metrics = [
        ("ROUGE-L",          "avg_rouge_l"),
        ("Judge Score",      "avg_judge_score"),
        ("  Relevance",      "avg_relevance"),
        ("  Factual",        "avg_factual"),
        ("  Completeness",   "avg_completeness"),
    ]
    for label, key in metrics:
        v1 = purell.get(key, 0)
        v2 = summary.get(key, 0)
        delta = v2 - v1
        sign  = "+" if delta >= 0 else ""
        print(f"{label:30s}  {v1:10.4f}  {v2:12.4f}  ({sign}{delta:.4f})")

    rag_hit = summary.get("source_hit_rate", 0)
    print(f"{'Source hit rate (RAG only)':30s}  {'N/A':>10}  {rag_hit*100:11.1f}%")

    print()
    print("GraphRAG results will be added in the final comparison notebook.")
    print("Add your GraphRAG per-question results in the same schema to complete the table.")
else:
    print("Baseline 1 results not found at /content/purell_qa_results.json")
    print("Run Baseline 1 first, or copy from Drive:")
    print("  shutil.copy('/content/drive/MyDrive/purell_qa_results.json', '/content/')")

## Summary

| Aspect | Choice | Reason |
|--------|--------|--------|
| LLM | Mistral NeMo 12B Instruct (4-bit NF4) | Same as Baseline 1 — context source is the only variable |
| Embedding | `all-MiniLM-L6-v2` on CPU | Preserves GPU VRAM for LLM; standard RAG benchmark model |
| Vector store | FAISS IndexFlatIP | Exact cosine search; no server required |
| Chunk size | 512 chars / 64 overlap | Preserves sentence/paragraph coherence |
| Splitter | RecursiveCharacterTextSplitter | Paragraph-first splitting preserves semantic coherence |
| Retrieval query | The question itself | Most direct semantic match for QA |
| Top-k | 5 chunks | ~2500 chars context; fits in model window |
| QA set | Shared `qa_eval_set.json` from Baseline 1 | All 3 baselines answer identical questions |
| ROUGE-L | Automated overlap | Fast, reproducible baseline metric |
| LLM-as-Judge | 3-dimension scoring | Captures quality beyond lexical overlap |
| Source hit rate | Retrieval-specific metric | Measures whether RAG found the right article |

**Output files for the final three-way comparison:**
- `/content/rag_qa_results.json` — answers + full scores
- `/content/rag_faiss.index` — saved index (reusable)

**What to expect vs GraphRAG:**  
Vanilla RAG typically outperforms Pure LLM on event-specific factual questions (when the retriever finds the right article).  
GraphRAG should outperform Vanilla RAG on questions requiring multi-hop reasoning across entities — e.g., "What companies did Marriott compete with during their 2011 profit cut?" — because the knowledge graph explicitly stores those relationships, while Vanilla RAG can only retrieve flat text and hope the relevant connections appear within a single chunk.